# 04 — Feature Engineering

Every target-derived feature is computed **as of the forecast origin**, never as of the target date.

To forecast day `origin+14`, the model may use demand up to `origin`. It may not use a lag measured from the target date, because that would be `origin+13` — a day that has not happened yet. Building `rolling_mean_7` with a panel-wide groupby-shift is the classic way this goes wrong, and it inflates reported accuracy substantially.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

panel: 1,003,600 rows, 600 series, 1969 days


In [2]:
from src.data.loader import rolling_origin_folds
from src.features.build import build_supervised, origin_features, feature_columns
fold = rolling_origin_folds(cfg, panel)[-1]
df = build_supervised(cfg, panel, fold)
print(f'{len(df):,} rows, {len(feature_columns(df))} features')
print('lags:', cfg.lags, '| rolling windows:', cfg.rolling_windows)
df.head(3)

16,800 rows, 54 features
lags: [1, 7, 14, 28] | rolling windows: [7, 14, 28]


,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_max_7,nonzero_rate_7,rolling_mean_14,rolling_std_14,...,doy_sin,doy_cos,product_age_days,discount_vs_28d,price_ratio_vs_origin,sku_id,dept_id,cat_id,store_id,state_id
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.911169,-0.412034,1509,-1.192093e-07,1.0,FOODS_1_004,FOODS_1,FOODS,CA_1,CA
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.911169,-0.412034,1509,-1.192093e-07,1.0,FOODS_1_004,FOODS_1,FOODS,TX_1,TX
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.911169,-0.412034,1509,-1.192093e-07,1.0,FOODS_1_004,FOODS_1,FOODS,WI_1,WI


## Origin-anchored features for one origin

In [3]:
f = origin_features(panel[panel.date <= fold.origin], fold.origin, cfg)
f[[c for c in f.columns if c.startswith(('lag_','rolling_mean_'))]].head()

,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28
series_id,,,,,,,
FOODS_1_004_CA_1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
FOODS_1_004_TX_1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
FOODS_1_004_WI_1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
FOODS_1_031_CA_1,3.0,1.0,0.0,1.0,1.714286,1.000000,0.857143
FOODS_1_031_TX_1,4.0,0.0,1.0,2.0,1.000000,1.571429,1.571429


## Proof that features cannot see the future

Overwrite every actual after the origin with nonsense, rebuild, and compare.

In [4]:
from src.data.audit import leakage_probe
leakage_probe(cfg, panel, fold)

{'fold': 'fold_4',
 'origin': '2016-04-24',
 'n_features_checked': 54,
 'n_rows_compared': 1800,
 'features_changed_by_future_mutation': [],
 'leak_free': True}

## What is allowed to come from the future

`sell_price`, `snap` and the calendar for the **target date**. A retailer sets prices and promotion calendars in advance, so these are genuinely known at forecast time. That is an assumption about the business process, declared in `configs/config.yaml` under `features.assume_known_future`.

In [5]:
cfg['features']

{'lags': [1, 7, 14, 28],
 'rolling_windows': [7, 14, 28],
 'assume_known_future': ['sell_price', 'snap', 'calendar']}